In [72]:
!pip install -q gradio groq langchain langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf


In [73]:
import os
from google.colab import userdata
os.environ['groq']=userdata.get('groq')
print("key is set")

key is set


In [ ]:
from groq import Groq
client=Groq(api_key=os.environ['groq'])
response=client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a expert football pundit"},
        {"role": "user", "content": "give me predictions and reasons for the champions for fifa"},
    ]
)
print(response.choices[0].message.content)

In [80]:
import gradio as gr
initial_system_prompt = '''You have indepth knowledge about formula 1 and everything related to it like cars,drivers,teams,logistics,budgets etc.
The level of your knowledge and game sense transcends team principals,drivers,pundits '''
def respond(message, history, system_prompt_ip, temperature_ip):
  messages = []
  if system_prompt_ip:
      messages.append({"role": "system", "content": system_prompt_ip})

  for chat_pair in history:
    if not isinstance(chat_pair, (list, tuple)):
        continue
    human_msg = chat_pair[0] if len(chat_pair) > 0 else ""
    ai_msg = chat_pair[1] if len(chat_pair) > 1 else ""

    if human_msg:
        messages.append({"role": "user", "content": human_msg})
    if ai_msg is not None and ai_msg != "":
        messages.append({"role": "assistant", "content": ai_msg})

  messages.append({"role": "user", "content": message})

  response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    temperature=temperature_ip,
    stream=True
  )

  partial = ""
  for chunk in response:
    if chunk.choices[0].delta.content is not None:
      partial += chunk.choices[0].delta.content
      yield partial


system_prompt_textbox = gr.Textbox(label="System Prompt", value=initial_system_prompt, lines=5)
temperature_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.6, step=0.1, label="Temperature")

demo = gr.ChatInterface(
    fn=respond,
    type="messages",
    textbox=gr.Textbox(placeholder="Type Here", container=False, scale=7),
    title="F1 Chatbot",
    cache_examples=False,
    additional_inputs=[system_prompt_textbox, temperature_slider]
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6237237789b6a3f7c8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6237237789b6a3f7c8.gradio.live
